# OpenAaaS 快速入门 / OpenAaaS Quick Start

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Wolido/OpenAaaS/main?filepath=binder%2Fquickstart.ipynb)

在浏览器中直接体验 OpenAaaS Python SDK，无需本地安装。
Run OpenAaaS Python SDK directly in your browser, no local installation needed.

## 简介 / Introduction

**OpenAaaS**（Open Agent-as-a-Service）是一个面向科学研究的智能体编排平台。
**OpenAaaS** (Open Agent-as-a-Service) is an agent orchestration platform for scientific research.

本 Notebook 将带你完成一次完整的科研任务流程：注册账号 → 发现服务 → 提交任务 → 获取并展示结果。
This notebook walks you through a complete research workflow: register → discover services → submit a task → retrieve and display results.

---

每一步都配有代码示例，点击上方的 **Binder** 徽章即可在浏览器中直接运行。
Each step includes runnable code examples. Click the **Binder** badge above to run everything in your browser.

In [ ]:
# 安装 pyopenaaas
# Install pyopenaaas
!pip install pyopenaaas -q

In [ ]:
import pyopenaaas

# 创建客户端，默认连接公共服务器 https://api.open-aaas.com
# Create client, default public server
client = pyopenaaas.Client()

In [ ]:
# 注册获取 API Key（无需预先准备，自动生成随机用户名）
# Register to get an API Key (auto-generated random username)
result = client.register()
print("API Key:", result["api_key"])

In [ ]:
# 查看公共服务器上有哪些科研服务
# Discover available scientific services
services = client.list_services()
for svc in services:
    print(f"- {svc.name} (ID: {svc.id})")

In [ ]:
# 获取第一个服务的详细用法说明
# Get detailed usage for the first service
usage = client.get_service_usage(services[0].id)
print(usage.usage)

In [ ]:
# 提交一个科研任务
# Submit a research task
task = client.submit_task(
    service_id=services[0].id,
    # task_prompt="Research how to design high-entropy alloys with high-temperature ductility and oxidation resistance",
    task_prompt="为我调研一下如何设计具有高温塑性且抗氧化的高熵合金",
    output_prompt="",
)
print(f"Task ID: {task.id}, Status: {task.status}")

In [ ]:
import time
from IPython.display import clear_output

# 自动轮询等待任务完成（每10秒检查一次）
# Poll until task completes (check every 10 seconds)
start_time = time.time()
while True:
    task = client.get_task(task.id)
    elapsed = time.time() - start_time
    if task.status in ("completed", "failed"):
        clear_output(wait=True)
        print(f"✅ Task completed after {elapsed:.1f}s")
        print(f"Final status: {task.status}")
        break
    clear_output(wait=True)
    print(f"⏳ Waiting for task to complete...")
    print(f"   Elapsed: {elapsed:.1f}s  |  Status: {task.status}")
    time.sleep(10.0)

In [ ]:
# 下载所有结果文件到本地 .OpenAaaS/downloads/<task_id>/
# Download all result files
if task.is_success():
    paths = client.download_all_files(task.id)
    for p in paths:
        print(f"Saved: {p}")
else:
    print(f"Task failed: {task.error_message}")

In [ ]:
from IPython.display import Markdown, display

# 找到下载目录中的 response.md 文件并直接渲染展示
# Find the downloaded response.md and render it directly
download_dir = pyopenaaas._utils._get_download_dir(task.id)
md_files = list(download_dir.glob("*.md"))

if md_files:
    for md_file in md_files:
        print(f"\n{'='*60}")
        print(f"📄 {md_file.name}")
        print(f"{'='*60}\n")
        display(Markdown(md_file.read_text(encoding="utf-8")))
else:
    print("No .md files found in download directory.")

## 总结 / Summary

OpenAaaS 是一个面向 **AI for Science** 的智能体编排网络（Agent Orchestration Network）。
刚才你通过 Python SDK 体验了一次完整的科研任务流程——注册、发现服务、提交任务、获取结果。

但 OpenAaaS 的设计初衷是让**任意 Agent** 都能方便地接入和使用科研能力。对于日常使用，我们更推荐以下接入方式：

- **桌面客户端** — 图形界面，下载即用，适合非技术用户
- **MCP 适配器** — Claude Desktop、Cursor、Cline 等支持 MCP 的客户端，一条配置即可接入
- **pi 插件** — 在对话中直接调用，零配置

📖 详情与安装指南：https://github.com/Wolido/OpenAaaS#如何使用

---

OpenAaaS is an **Agent Orchestration Network for AI for Science**. You just experienced a complete research workflow via the Python SDK — register, discover services, submit tasks, and retrieve results.

But OpenAaaS is designed so that **any agent** can easily access and use scientific capabilities. For daily use, we recommend:

- **Desktop Client** — GUI, download and run, ideal for non-technical users
- **MCP Adapter** — For Claude Desktop, Cursor, Cline, etc. One config to connect
- **pi Plugin** — Invoke directly in chat, zero configuration

📖 Details & installation: https://github.com/Wolido/OpenAaaS#how-to-use